# 02. Chat Model 호출

LangChain에서 OpenAI ChatModel을 불러오고 기본 호출 방법을 실습합니다.

## 1. 환경 설정

In [12]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)
if not os.getenv("LLM_API_KEY"):
    raise ValueError("프로젝트 .env에 LLM_API_KEY를 설정하세요.")


## 2. ChatOpenAI 초기화 (MonoRouter)

- `model`: 사용할 모델 이름
- `temperature`: 0에 가까울수록 일관된 답변, 1에 가까울수록 창의적인 답변

In [13]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.getenv("LLM_BASE_URL", "https://monogpt.kr/api/monorouter/v1"), use_responses_api=False, model="gpt-4.1", temperature=0)
print(llm)

metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain-openai': '1.6.7'}} profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x11157d670> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11157c2c0> root_client=<openai.OpenAI object at 0x10d31c110> root_async_client=<openai.AsyncOpenAI object at 0x10d287e30>

## 3. 기본 호출 - invoke()

문자열을 직접 전달하면 HumanMessage로 자동 변환됩니다.

In [14]:
response = llm.invoke("LangChain이 무엇인지 한 문장으로 설명해줘")
print(response)

content='LangChain은 다양한 언어 모델과 데이터 소스를 연결하여 복잡한 자연어 처리 애플리케이션을 쉽게 개발할 수 있도록 도와주는 오픈소스 프레임워크입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 19, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1', 'system_fingerprint': 'fp_cf94e8683e', 'id': 'chatcmpl-EU3II8fDnwZhHfdTQUFMM4nP299gr', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0f5db-abc2-7dd3-98ad-76bfa1163d47-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 19, 'output_tokens': 43, 'total_tokens': 62, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'r

In [15]:
# content만 출력
print(response.content)

LangChain은 다양한 언어 모델과 데이터 소스를 연결하여 복잡한 자연어 처리 애플리케이션을 쉽게 개발할 수 있도록 도와주는 오픈소스 프레임워크입니다.


## 4. 메시지 객체로 호출

역할(role)을 구분하여 메시지를 전달할 수 있습니다.
- `SystemMessage`: AI의 역할/페르소나 설정
- `HumanMessage`: 사용자 입력

In [16]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="당신은 친절한 파이썬 튜터입니다."),
    HumanMessage(content="리스트 컴프리헨션을 초보자에게 설명해줘")
]

response = llm.invoke(messages)
print(response.content)

물론이죠! 리스트 컴프리헨션(list comprehension)은 파이썬에서 리스트를 쉽고 간단하게 만드는 방법입니다. 초보자도 이해할 수 있도록 쉽게 설명해드릴게요.

### 1. 기본적인 리스트 만들기

예를 들어, 1부터 5까지의 숫자를 담은 리스트를 만들고 싶다고 해볼게요.

**기존 방법:**
```python
numbers = []
for i in range(1, 6):
    numbers.append(i)
print(numbers)  # [1, 2, 3, 4, 5]
```

**리스트 컴프리헨션 사용:**
```python
numbers = [i for i in range(1, 6)]
print(numbers)  # [1, 2, 3, 4, 5]
```

### 2. 리스트 컴프리헨션 구조

리스트 컴프리헨션의 기본 구조는 이렇습니다:

```python
[표현식 for 변수 in 반복가능한객체]
```

- **표현식**: 리스트에 담길 값 (예: i, i*2 등)
- **변수**: 반복문에서 사용할 변수 (예: i)
- **반복가능한객체**: 리스트, range, 문자열 등 반복할 수 있는 것

### 3. 조건문도 넣을 수 있어요!

예를 들어, 1부터 10까지 중에서 짝수만 리스트로 만들고 싶다면?

```python
evens = [i for i in range(1, 11) if i % 2 == 0]
print(evens)  # [2, 4, 6, 8, 10]
```

### 4. 한눈에 보기

- **for문으로 리스트 만들기** → 코드가 길어짐
- **리스트 컴프리헨션** → 한 줄로 간단하게!

### 5. 연습 문제

1부터 5까지의 제곱을 리스트로 만들어볼까요?

```python
squares = [i**2 for i in range(1, 6)]
print(squares)  # [1, 4, 9, 16, 25]
```

---

**정리:**  
리스트 컴프리헨션은 "for문을 한 줄로 줄여서 리스트를 만드는 방법"입니다.  
코

- 마크다운으로 출력해서 보기

In [17]:
from IPython.display import Markdown, display
display(Markdown(response.content))

물론이죠! 리스트 컴프리헨션(list comprehension)은 파이썬에서 리스트를 쉽고 간단하게 만드는 방법입니다. 초보자도 이해할 수 있도록 쉽게 설명해드릴게요.

### 1. 기본적인 리스트 만들기

예를 들어, 1부터 5까지의 숫자를 담은 리스트를 만들고 싶다고 해볼게요.

**기존 방법:**
```python
numbers = []
for i in range(1, 6):
    numbers.append(i)
print(numbers)  # [1, 2, 3, 4, 5]
```

**리스트 컴프리헨션 사용:**
```python
numbers = [i for i in range(1, 6)]
print(numbers)  # [1, 2, 3, 4, 5]
```

### 2. 리스트 컴프리헨션 구조

리스트 컴프리헨션의 기본 구조는 이렇습니다:

```python
[표현식 for 변수 in 반복가능한객체]
```

- **표현식**: 리스트에 담길 값 (예: i, i*2 등)
- **변수**: 반복문에서 사용할 변수 (예: i)
- **반복가능한객체**: 리스트, range, 문자열 등 반복할 수 있는 것

### 3. 조건문도 넣을 수 있어요!

예를 들어, 1부터 10까지 중에서 짝수만 리스트로 만들고 싶다면?

```python
evens = [i for i in range(1, 11) if i % 2 == 0]
print(evens)  # [2, 4, 6, 8, 10]
```

### 4. 한눈에 보기

- **for문으로 리스트 만들기** → 코드가 길어짐
- **리스트 컴프리헨션** → 한 줄로 간단하게!

### 5. 연습 문제

1부터 5까지의 제곱을 리스트로 만들어볼까요?

```python
squares = [i**2 for i in range(1, 6)]
print(squares)  # [1, 4, 9, 16, 25]
```

---

**정리:**  
리스트 컴프리헨션은 "for문을 한 줄로 줄여서 리스트를 만드는 방법"입니다.  
코드를 짧고 읽기 쉽게 만들어줘요!

궁금한 점이 있으면 언제든 질문해 주세요 😊

## 5. 스트리밍 호출 - stream()

- 토큰이 생성되는 즉시 출력함.
- llm 객체에 질문을 보내고, 응답을 한 번에 받지 않고 조각(chunk) 단위로 순차적으로 받음.
- end=""를 쓰면 줄바꿈 없이 이어서 출력함.
- flush=True는 출력 버퍼를 즉시 비우라는 의미임.

In [18]:
for chunk in llm.stream("최신 langchain 버전을 알여주고, 특장점을 알여줘 "):
    print(chunk.content, end="", flush=True)

최신 LangChain 버전 (2024년 6월 기준)

2024년 6월 기준, **LangChain의 최신 버전은 `0.1.x`** (예: `0.1.16` 등)입니다.  
최신 버전은 [공식 PyPI 페이지](https://pypi.org/project/langchain/) 또는 [GitHub 릴리즈](https://github.com/langchain-ai/langchain/releases)에서 확인할 수 있습니다.

---

## LangChain의 주요 특장점 (2024년 기준)

### 1. **표준화된 LLM 파이프라인**
- 프롬프트 템플릿, LLM 호출, 출력 파싱 등 LLM 활용에 필요한 모든 과정을 표준화된 컴포넌트로 제공합니다.
- 다양한 LLM(오픈AI, Anthropic, HuggingFace 등)과 손쉽게 연동할 수 있습니다.

### 2. **체인(Chain)과 에이전트(Agent) 구조**
- 여러 단계를 연결한 체인(Chain) 구조로 복잡한 워크플로우를 쉽게 구현할 수 있습니다.
- 에이전트(Agent)는 도구(Tool)와 외부 API를 활용해 동적으로 문제를 해결할 수 있습니다.

### 3. **풍부한 통합(Integration)**
- 벡터DB(Chroma, Pinecone, FAISS 등), 검색엔진, 데이터베이스, 클라우드 서비스 등과 쉽게 연동할 수 있습니다.
- Retrieval-Augmented Generation(RAG) 구현이 매우 용이합니다.

### 4. **문서 처리 및 검색(RAG) 지원**
- 문서 분할, 임베딩, 검색, 요약 등 RAG 파이프라인을 위한 다양한 유틸리티 제공.
- 최신 버전에서는 `langchain.text_splitter`, `langchain.embeddings`, `langchain.vectorstores` 등 모듈이 강화되었습니다.

### 5. **유연한 커스터마이징**
- 체인, 프롬프트, 출력 파서 등 모든 컴포넌트를 자유롭게 커스터마이징 가능.
- 파이프라인을 코드로 명확

## 6. 응답 메타데이터 확인

토큰 사용량 등 메타정보를 확인할 수 있습니다.

In [19]:
response = llm.invoke("안녕")
print("content:", response.content)
print("usage:", response.usage_metadata)

content: 안녕하세요! 무엇을 도와드릴까요? 😊
usage: {'input_tokens': 9, 'output_tokens': 12, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
